In [ ]:
df = pd.read_csv('positives_grid_1h_2h.csv')
print("양성:", (df['is_hardcase']==1).sum())
print("정상로그:", (df['is_hardcase']==0).sum())
# 정상로그가 양성의 4배 이상이면 1:4 문제없음
# 표본 개수 충분. 비교 가능.

양성: 14819
정상로그: 1845858


In [12]:
import pandas as pd 
import numpy as np
from sklearn.metrics import roc_auc_score

#창 안에서 음성 뽑음.
df = pd.read_csv("positives_grid_1h_2h.csv")

df = df.sort_values(['node', 'grid_id_1h', 'timestamp'])

df['rank'] = df.groupby(['node', 'grid_id_1h']).cumcount()
df['n'] = df.groupby(['node', 'grid_id_1h'])['line_id'].transform('size')

df = df[df['n'] >= 2]

df['position'] = df['rank'] / (df['n'] - 1)

def matching_auc(df, ratio, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    
    for (node, grid), g in df.groupby(['node', 'grid_id_1h']):
        pos = g[g['is_hardcase'] == 1]
        neg = g[g['is_hardcase'] == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        need = len(pos) * ratio
        if len(neg) >= need:
            take = neg.sample(need, random_state = rng.integers(1e9))
        else:
            take = neg
        rows.append(pd.concat([pos, take]))
    matched = pd.concat(rows)
    auc = roc_auc_score(matched['is_hardcase'], matched['position'])
    return auc, len(matched[matched['is_hardcase'] == 1]), len(matched[matched['is_hardcase'] == 0])


for ratio in [1,4]:
    auc, npos, nneg = matching_auc(df, ratio)
    print(f"1:{ratio} AUC = {auc:.4f} |0.5(auc 기준) 차이| = {abs(auc-0.5):.4f} 양성개수 = {npos} 음성개수 = {nneg}")

1:1 AUC = 0.3508 |0.5(auc 기준) 차이| = 0.1492 양성개수 = 14724 음성개수 = 14723
1:4 AUC = 0.3491 |0.5(auc 기준) 차이| = 0.1509 양성개수 = 14724 음성개수 = 56465


In [10]:
def matched_permutation(df, ratio, n_perm=5000, seed = 42):
    rng = np.random.default_rng(seed)
    rows = []
    for (node, grid), g in df.groupby(['node', 'grid_id_1h']):
        pos = g[g['is_hardcase'] == 1]
        neg = g[g['is_hardcase'] == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        need = len(pos) * ratio
        take = neg.sample(need, random_state = rng.integers(1e9)) if len(neg) >= need else neg
        rows.append(pd.concat([pos, take]))
    matched = pd.concat(rows)
    
    y = matched['is_hardcase'].values
    score = matched['position'].values
    auc_real  = roc_auc_score(y, score)
    
    rng2 = np.random.default_rng(seed)
    count = 0
    for _ in range(n_perm):
        y_shuf = rng2.permutation(y)
        auc_perm = roc_auc_score(y_shuf, score)
        if abs(auc_perm - 0.5) >= abs(auc_real - 0.5):
            count += 1
    p = (count + 1) / (n_perm + 1)
    return auc_real, p

for ratio in [1,4]:
    auc, p = matched_permutation(df, ratio)
    print(f"1:{ratio} AUC = {auc:.4f} p = {p:.4f} {'신호 있음' if p<0.05 else '무의미'}")

1:1 AUC = 0.3508 p = 0.0002 신호 있음
1:4 AUC = 0.3491 p = 0.0002 신호 있음


오히려 음성을 더 많이 집어 넣어도, 0.5에서 멀어지고 신호가 강해짐.

Go 판정 결정.